# 02. Visualizaciones del proyecto

## Presentación de la sección
Este cuaderno desarrolla las visualizaciones planteadas en el informe. La secuencia respeta las cuatro secciones del trabajo: Resumen General, Competencia, Riesgo Económico y Transparencia Geográfica. Además, se distribuye el uso de pandas, matplotlib, seaborn y plotly para evidenciar distintas formas de representación gráfica dentro del análisis académico.


In [ ]:
import importlib.util
requeridas = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']
faltantes = [pkg for pkg in requeridas if importlib.util.find_spec(pkg) is None]
if faltantes:
    raise ModuleNotFoundError(f'Faltan librerias en este kernel: {faltantes}. Instale requirements.txt o seleccione otro interprete.')
print('Entorno verificado correctamente.')


## Librerias de trabajo


In [ ]:
from pathlib import Path
import importlib.util
import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

renderer_disponible = 'browser'
if importlib.util.find_spec('IPython') and importlib.util.find_spec('nbformat'):
    renderer_disponible = 'notebook_connected'
pio.renderers.default = renderer_disponible
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'data').exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / 'data').exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError('No se encontr? la carpeta data del proyecto.')

MASTER_FILE = PROJECT_DIR / 'data' / 'contrataciones_peru_2022_2024_maestro.csv'
df = pd.read_csv(MASTER_FILE, low_memory=False)

def obtener_columna_anio(dataframe):
    for columna in dataframe.columns:
        normal = unicodedata.normalize('NFKD', str(columna)).encode('ascii', 'ignore').decode('ascii').lower()
        if normal in ('anio', 'ano'):
            return columna
    raise KeyError('No se encontr? una columna de a?o en la base.')

def es_valor_visible(serie):
    invalidos = {'sin dato', 'no especificado', 'nan', ''}
    return ~serie.astype(str).str.strip().str.lower().isin(invalidos)

col_anio = obtener_columna_anio(df)
for columna in ['categoria', 'metodo_simple', 'departamento', 'entidad_compradora', 'proveedor_ganador']:
    if columna in df.columns:
        df[columna] = df[columna].fillna('No especificado').astype(str).str.strip()
df['monto_adjudicado'] = pd.to_numeric(df['monto_adjudicado'], errors='coerce').fillna(0)
df['monto_MM'] = df['monto_adjudicado'] / 1_000_000
df['n_postores'] = pd.to_numeric(df['n_postores'], errors='coerce').fillna(0)
df['un_solo_postor'] = df['n_postores'].eq(1)
df['directa'] = df['metodo_simple'].astype(str).str.contains('direct', case=False, na=False)
col_fecha = 'fecha_proceso' if 'fecha_proceso' in df.columns else 'fecha'
df['fecha'] = pd.to_datetime(df[col_fecha], errors='coerce')
df['periodo_mensual'] = df['fecha'].dt.to_period('M').astype(str)
mes_num = df['fecha'].dt.month.fillna(1).astype(int).astype(str).str.zfill(2)
df['mes_reporte'] = pd.to_datetime(df[col_anio].astype(int).astype(str) + '-' + mes_num + '-01', errors='coerce')
print('Base lista para visualizaci?n:', df.shape)


# Sección 1. Resumen General

La primera sección presenta la magnitud del universo analizado y la composición general del gasto adjudicado.


## Indicador 1. Total de procesos analizados

**Relevancia:** establece el tamaño del universo evaluado y dimensiona el alcance del estudio.


In [ ]:
print('Procesos únicos analizados:', format(df['ocid'].nunique(), ','))


## Indicador 2. Monto adjudicado total

**Relevancia:** resume el peso económico agregado del conjunto de contrataciones revisadas.


In [ ]:
print('Monto adjudicado total (MM PEN):', round(df['monto_MM'].sum(), 2))


## Gráfico 1. Monto adjudicado por categoría

**Relevancia:** identifica en que tipo de contratación se concentra el gasto.

**Libreria utilizada:** pandas con backend gráfico de matplotlib.


In [ ]:
g1 = (df[es_valor_visible(df['categoria'])]
      .groupby('categoria', as_index=True)['monto_MM']
      .sum()
      .sort_values())
ax = g1.plot(kind='barh', color='steelblue')
ax.set_title('Monto adjudicado por categor?a')
ax.set_xlabel('Millones de PEN')
ax.set_ylabel('Categor?a')
plt.tight_layout()
plt.show()


## Gráfico 2. Procesos por año

**Relevancia:** compara la cantidad de procesos registrados en cada periodo.

**Libreria utilizada:** seaborn.


In [ ]:
g2 = df.groupby(col_anio, as_index=False).agg(procesos=('ocid', 'nunique'))
sns.barplot(data=g2, x=col_anio, y='procesos', color='steelblue')
plt.title('Cantidad de procesos por año')
plt.xlabel('Año')
plt.ylabel('Procesos')
plt.tight_layout()
plt.show()


## Gráfico 3. Participación de procesos por método

**Relevancia:** muestra la composición relativa de modalidades de contratación.

**Librería utilizada:** matplotlib.


In [ ]:
g3 = (df[es_valor_visible(df['metodo_simple'])]
      .groupby('metodo_simple', as_index=False)
      .agg(procesos=('ocid', 'nunique')))
plt.figure(figsize=(7, 7))
plt.pie(g3['procesos'], labels=g3['metodo_simple'], autopct='%1.1f%%', startangle=90)
plt.title('Participación de procesos por método')
plt.tight_layout()
plt.show()


## Gráfico 4. Distribución del monto por año y categoría

**Relevancia:** permite observar si el patrón de gasto cambia entre periodos.

**Librería utilizada:** plotly.


In [ ]:
g4 = (df[es_valor_visible(df['categoria'])]
      .groupby([col_anio, 'categoria'], as_index=False)
      .agg(monto_MM=('monto_MM', 'sum')))
fig = px.bar(g4, x=col_anio, y='monto_MM', color='categoria', barmode='group',
             color_discrete_map={'Bienes': '#61d6a3', 'Servicios': '#6ea8fe', 'Obras': '#ff6658'},
             title='Monto adjudicado por a?o y categor?a')
fig.update_layout(xaxis_title='A?o', yaxis_title='Monto adjudicado (MM PEN)')
fig.show()


# Sección 2. Competencia

La segunda sección evalúa la concurrencia de postores y la presencia de procesos con competencia reducida.


## Gráfico 5. Distribución del número de postores

**Relevancia:** describe el comportamiento general de la concurrencia en los procesos.

**Librería utilizada:** matplotlib.


In [ ]:
plt.hist(df.loc[df['n_postores'] > 0, 'n_postores'].clip(upper=10), bins=10, color='salmon', edgecolor='black')
plt.title('Distribución del número de postores por proceso')
plt.xlabel('Número de postores (10 representa 10 o más)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()


## Gráfico 6. Porcentaje de procesos con un solo postor por año

**Relevancia:** resume la principal señal de baja competencia por periodo.

**Librería utilizada:** pandas con backend gráfico de matplotlib.


In [ ]:
g6 = df.groupby(col_anio, as_index=True)['un_solo_postor'].mean().mul(100)
ax = g6.plot(kind='bar', color='indianred')
ax.set_title('Porcentaje de procesos con un solo postor por año')
ax.set_xlabel('Año')
ax.set_ylabel('Porcentaje')
plt.tight_layout()
plt.show()


## Gráfico 7. Top 15 departamentos con un solo postor

**Relevancia:** localiza territorialmente la mayor concentración de menor competencia.

**Librería utilizada:** seaborn.


In [ ]:
g7 = (df[df['un_solo_postor']].groupby('departamento', as_index=False).agg(procesos=('ocid', 'nunique')).sort_values('procesos', ascending=False).head(15))
sns.barplot(data=g7, x='procesos', y='departamento', color='darkorange')
plt.title('Top 15 departamentos con un solo postor')
plt.xlabel('Procesos')
plt.ylabel('Departamento')
plt.tight_layout()
plt.show()


## Gráfico 8. Distribución de postores por categoría

**Relevancia:** compara el comportamiento competitivo típico entre categorías.

**Librería utilizada:** seaborn.


In [ ]:
g8 = df[df['n_postores'] > 0].copy()
sns.boxplot(data=g8, x='categoria', y='n_postores', showfliers=False)
plt.title('Distribución de postores por categoría')
plt.xlabel('Categoría')
plt.ylabel('Número de postores')
plt.tight_layout()
plt.show()


# Sección 3. Riesgo Económico

La tercera sección cuantifica el peso monetario asociado a procesos con señales de menor competencia.


## Gráfico 9. Monto en riesgo frente a monto competitivo por año

**Relevancia:** separa la exposición económica con base en el nivel de competencia observado.

**Librería utilizada:** matplotlib.


In [ ]:
g9 = df.groupby(col_anio, as_index=False).agg(monto_total=('monto_adjudicado', 'sum'), monto_riesgo=('monto_adjudicado', lambda s: s[df.loc[s.index, 'un_solo_postor']].sum()))
g9['monto_competitivo'] = g9['monto_total'] - g9['monto_riesgo']
g9[['monto_riesgo', 'monto_competitivo']] = g9[['monto_riesgo', 'monto_competitivo']] / 1_000_000
x = np.arange(len(g9))
width = 0.35
plt.bar(x - width/2, g9['monto_riesgo'], width=width, label='Monto en riesgo')
plt.bar(x + width/2, g9['monto_competitivo'], width=width, label='Monto competitivo')
plt.xticks(x, g9[col_anio].astype(str))
plt.title('Monto en riesgo frente a monto competitivo por año')
plt.xlabel('Año')
plt.ylabel('Millones de PEN')
plt.legend()
plt.tight_layout()
plt.show()


## Gráfico 10. Evolución del gasto por categoría

**Relevancia:** permite identificar cambios en la importancia relativa de cada categoría.

**Librería utilizada:** plotly.


In [ ]:
g10 = df.groupby([col_anio, 'categoria'], as_index=False).agg(monto_MM=('monto_MM', 'sum'))
fig = px.line(g10, x=col_anio, y='monto_MM', color='categoria', markers=True, title='Evolución anual del gasto por categoría')
fig.update_layout(xaxis_title='Año', yaxis_title='Millones de PEN')
fig.show()


## Gráfico 11. Top 15 entidades con más contrataciones directas

**Relevancia:** identifica concentración institucional de procedimientos directos.

**Librería utilizada:** pandas con backend gráfico de matplotlib.


In [ ]:
g11 = (df[df['directa']].groupby('entidad_compradora', as_index=True)['ocid'].nunique().sort_values(ascending=False).head(15).sort_values())
ax = g11.plot(kind='barh', color='slateblue')
ax.set_title('Top 15 entidades con más contrataciones directas')
ax.set_xlabel('Procesos directos')
ax.set_ylabel('Entidad compradora')
plt.tight_layout()
plt.show()


## Gráfico 12. Monto adjudicado frente a número de postores

**Relevancia:** contrasta dimensión económica y nivel de concurrencia.

**Librería utilizada:** plotly.


In [ ]:
g12 = df[['ocid', 'n_postores', 'monto_adjudicado', 'un_solo_postor', 'categoria', 'departamento']].dropna().copy()
fig = px.scatter(g12, x='n_postores', y='monto_adjudicado', color='un_solo_postor', title='Monto adjudicado frente a número de postores', opacity=0.5)
fig.update_layout(xaxis_title='Número de postores', yaxis_title='Monto adjudicado (PEN)')
fig.show()


# Sección 4. Transparencia Geográfica

La cuarta sección traslada el análisis al territorio y consolida indicadores comparables entre departamentos.


## Gráfico 13. Riesgo por departamento y categoría

**Relevancia:** resume el porcentaje de procesos con un solo postor según territorio y categoría.

**Librería utilizada:** seaborn.


In [ ]:
top_dptos = (df[es_valor_visible(df['departamento'])]
             .groupby('departamento')['ocid']
             .nunique()
             .sort_values(ascending=False)
             .head(15)
             .index)
g13 = (df[df['departamento'].isin(top_dptos) & es_valor_visible(df['categoria'])]
       .groupby(['departamento', 'categoria'])['un_solo_postor']
       .mean()
       .mul(100)
       .reset_index()
       .pivot(index='departamento', columns='categoria', values='un_solo_postor')
       .fillna(0))
plt.figure(figsize=(11, 8))
sns.heatmap(g13, cmap='YlOrRd', annot=True, fmt='.1f')
plt.title('Riesgo por departamento y categor?a (%)')
plt.xlabel('Categor?a')
plt.ylabel('Departamento')
plt.tight_layout()
plt.show()


## Gráfico 14. Evolución mensual de directas frente a competitivas

**Relevancia:** permite observar la trayectoria temporal del método de contratación.

**Librería utilizada:** matplotlib con apoyo de seaborn.


In [ ]:
g14 = (df[df['metodo_simple'].isin(['Competitivo', 'Directa']) & df['mes_reporte'].notna()]
       .groupby(['mes_reporte', 'metodo_simple'], as_index=False)
       .agg(procesos=('ocid', 'nunique')))
g14 = g14.sort_values('mes_reporte')
plt.figure(figsize=(12, 6))
sns.lineplot(data=g14, x='mes_reporte', y='procesos', hue='metodo_simple', marker='o', hue_order=['Competitivo', 'Directa'])
plt.title('Evolución mensual de contrataciones directas y competitivas')
plt.xlabel('Periodo mensual del reporte')
plt.ylabel('Procesos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## Gráfico 15. Top 15 proveedores ganadores repetidos

**Relevancia:** muestra la concentración de adjudicaciones en un grupo reducido de proveedores.

**Librería utilizada:** pandas con backend gráfico de matplotlib.


In [ ]:
g15 = (df[es_valor_visible(df['proveedor_ganador'])]
       .groupby('proveedor_ganador', as_index=True)['ocid']
       .nunique()
       .sort_values(ascending=False)
       .head(15)
       .sort_values())
ax = g15.plot(kind='barh', color='teal')
ax.set_title('Top 15 proveedores ganadores repetidos')
ax.set_xlabel('Procesos adjudicados')
ax.set_ylabel('Proveedor ganador')
plt.tight_layout()
plt.show()


## Gráfico 16. Score de transparencia por departamento

**Relevancia:** sintetiza en un solo indicador la participación de contratación directa, la presencia de un solo postor y la proporción de monto en riesgo. En esta versión, el resultado se presenta de forma gráfica para facilitar la comparación territorial.

**Librería utilizada:** plotly.


In [ ]:
g16 = (df[es_valor_visible(df['departamento'])]
       .groupby('departamento', as_index=False)
       .agg(pct_directa=('directa', 'mean'),
            pct_un_solo_postor=('un_solo_postor', 'mean'),
            monto_total=('monto_adjudicado', 'sum'),
            monto_riesgo=('monto_adjudicado', lambda s: s[df.loc[s.index, 'un_solo_postor']].sum())))
g16['pct_directa'] = g16['pct_directa'] * 100
g16['pct_un_solo_postor'] = g16['pct_un_solo_postor'] * 100
g16['pct_monto_riesgo'] = np.where(g16['monto_total'] > 0, (g16['monto_riesgo'] / g16['monto_total']) * 100, 0)
g16['score_transparencia'] = 100 - ((g16['pct_directa'] + g16['pct_un_solo_postor'] + g16['pct_monto_riesgo']) / 3)
g16 = g16.sort_values('score_transparencia', ascending=False).head(15).sort_values('score_transparencia')
fig = px.bar(g16, x='score_transparencia', y='departamento', orientation='h', color='score_transparencia',
             text='score_transparencia', color_continuous_scale=['#2b3350', '#6ea8fe', '#ff6658'],
             title='Score de transparencia por departamento')
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(xaxis_title='Score de transparencia', yaxis_title='Departamento')
fig.show()
